### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

## vector
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

In [25]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [26]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [27]:
## save the sample documents to files (temporary)
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt", 'w') as f:
        f.write(doc)
print(f"Sample documents created in {temp_dir}")

Sample documents created in C:\Users\hdhin\AppData\Local\Temp\tmprnjatc60


In [28]:
## save the sample documents to files to a permanent location

for i, doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt", 'w') as f:
        f.write(doc)
print(f"Sample documents created in {os.getcwd()}")

Sample documents created in c:\Personal\upskill2025\Udemy_RAG_github\2-Vector Stores


## Document Loading

In [29]:
from langchain_community.document_loaders import DirectoryLoader

# Load documents from a ddirectory
loader = DirectoryLoader(
    temp_dir, 
    glob = '*.txt',
    loader_cls = TextLoader,
    loader_kwargs = {'encoding' : 'utf-8'}

)
docs = loader.load()

print(f"Loaded {len(docs)} documents")
print(f"\n First Document preview : ")
print(f"{docs[0].page_content[:200]}...")

Loaded 3 documents

 First Document preview : 

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


In [30]:
docs

[Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of i

## Document Splitting

In [31]:
## Initialise text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, ## Max size of each chunk
    chunk_overlap = 50, ## Overlap between the chunks to maintain context
    length_function = len, 
    separators =["\n\n", "\n", ".", " ", ""]   ## Hierarchy of separators

)

chunks = text_splitter.split_documents(docs)

print(f"Created {len(chunks)} chunks from {len(docs)} documents")
print(f"\n Chunk Example : ")
print(f"Content : {chunks[0].page_content[:150]} ...")
print(f"Metadata : {chunks[0].metadata}")

Created 7 chunks from 3 documents

 Chunk Example : 
Content : Machine Learning Fundamentals ...
Metadata : {'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_0.txt'}


In [32]:
chunks

[Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_0.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_0.txt'}, page_content='interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_1.txt'}, page_content='Deep L

## Embedding Models

In [33]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [34]:
sample_text = 'machine learning is fascinating'
embeddings = OpenAIEmbeddings()
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000208871BA450>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000020887FDA750>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [35]:
vector = embeddings.embed_query(sample_text)
vector

[-0.03216513246297836,
 0.00909043475985527,
 0.01527246180921793,
 -0.026574082672595978,
 -0.004711222369223833,
 0.017105156555771828,
 0.0035790682304650545,
 0.013453046791255474,
 -0.020757267251610756,
 -0.03694607689976692,
 0.00361890927888453,
 0.03216513246297836,
 -0.01588336005806923,
 -0.012749184854328632,
 -0.0003799439873546362,
 0.01650753803551197,
 0.035591475665569305,
 0.005959580186754465,
 0.005358641967177391,
 -0.011308262124657631,
 -0.028712227940559387,
 0.02480779029428959,
 0.0030129910446703434,
 -0.043851885944604874,
 -0.018698804080486298,
 0.001402742345817387,
 0.004644820466637611,
 -0.03490089625120163,
 -0.020040124654769897,
 -0.006547237746417522,
 0.023320384323596954,
 -0.0015645972453057766,
 -0.011899239383637905,
 -0.039203744381666183,
 -0.0155247887596488,
 -0.017330924049019814,
 0.0004959399811923504,
 0.0016285092569887638,
 -0.015139657072722912,
 -0.0021298443898558617,
 0.01822070963680744,
 0.021022874861955643,
 -0.00300137070007

## Initialise the ChromaDB Vector Store and store the chunks in Vector Representation

In [36]:
## Create a ChromaDB vector store
persist_directory = "./chroma_db" ## where will the embeddings be stored and what will be the location of that vectorstore
"" \
## Initialise ChromaDB with OpenAI embeddings
vectorstore = Chroma.from_documents(
    documents = chunks, 
    embedding = OpenAIEmbeddings(),
    persist_directory = persist_directory,
    collection_name = 'rag_collection'
)

print(f"Vectorstore created with {vectorstore._collection.count()} vectors")
print(f"Persisted to : {persist_directory}")

Vectorstore created with 7 vectors
Persisted to : ./chroma_db


## Test the similarity Search 

In [37]:
query = 'What are the types of machine learning?'

similar_docs = vectorstore.similarity_search(query, k=3)
similar_docs

[Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_0.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_1.txt'}, page_content='Deep Learning and Neural Networks')]

In [38]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What are the types of machine learning?

Top 3 similar chunks:

--- Chunk 1 ---
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine l...
Source: C:\Users\hdhin\AppData\Local\Temp\tmprnjatc60\doc_0.txt

--- Chunk 2 ---
Machine Learning Fundamentals...
Source: C:\Users\hdhin\AppData\Local\Temp\tmprnjatc60\doc_0.txt

--- Chunk 3 ---
Deep Learning and Neural Networks...
Source: C:\Users\hdhin\AppData\Local\Temp\tmprnjatc60\doc_1.txt


### Advanced Similarity Search With Scores

In [39]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
  0.24218560755252838),
 (Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_0.txt'}, page_content='Machine Learning Fundamentals'),
  0.2677447497844696),
 (Document(metadata={'source': 'C:\\Users\\hdhin\\AppData\\Local\\Temp\\tmprnjatc60\\doc_1.txt'}, page_content='Deep Learning and Neural Networks'),
  0.2933101952075958)]

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

from langchain_openai import ChatOpenAI